In [1]:
import os
from pyspark.sql import SparkSession

# ---------- MinIO (S3A) ----------
DATA_LAKE_ENDPOINT = os.environ.get("DATA_LAKE_ENDPOINT", "http://minio:9000")
DATA_LAKE_ACCESS_KEY_ID = os.environ.get("DATA_LAKE_ACCESS_KEY_ID", "minioadmin")
DATA_LAKE_SECRET_ACCESS_KEY = os.environ.get("DATA_LAKE_SECRET_ACCESS_KEY", "minioadmin")
DATA_LAKE_REGION = os.environ.get("DATA_LAKE_REGION", "us-east-1")

# ---------- Mongo ----------
MONGO_HOST = os.environ.get("MONGO_HOST", "mongodb")
MONGO_PORT = os.environ.get("MONGO_PORT", "27017")
MONGO_AUTH_DB = os.environ.get("MONGO_AUTH_DB", "admin")
MONGO_USER = os.environ.get("MONGO_INITDB_ROOT_USERNAME")
MONGO_PASS = os.environ.get("MONGO_INITDB_ROOT_PASSWORD")

if not MONGO_USER or not MONGO_PASS:
    raise RuntimeError("Missing Mongo env vars: MONGO_INITDB_ROOT_USERNAME / MONGO_INITDB_ROOT_PASSWORD")

mongo_uri = f"mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:{MONGO_PORT}/{MONGO_AUTH_DB}?authSource=admin"

# ---------- Build Spark ----------
spark = (
    SparkSession.builder
    .appName("Yelp-Analytics-Jupyter")
    # ---- S3A / MinIO ----
    .config("spark.hadoop.fs.s3a.endpoint", DATA_LAKE_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", DATA_LAKE_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", DATA_LAKE_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    # (Optional) region; some stacks use it
    .config("spark.hadoop.fs.s3a.endpoint.region", DATA_LAKE_REGION)
    # ---- Delta (if you use it) ----
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # ---- Mongo Connector defaults ----
    .config("spark.mongodb.read.connection.uri", mongo_uri)
    .config("spark.mongodb.write.connection.uri", mongo_uri)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("✅ Spark ready")
print("MinIO endpoint:", DATA_LAKE_ENDPOINT)
print("Mongo URI host:", f"{MONGO_HOST}:{MONGO_PORT}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/23 10:52:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark ready
MinIO endpoint: http://minio:9000
Mongo URI host: mongodb:27017


In [2]:
!env

MONGO_HOST=mongodb
LANGUAGE=en_US:en
MONGO_JAVA_DRIVER_VERSION=5.6.2
MPLBACKEND=module://matplotlib_inline.backend_inline
HOSTNAME=72c7bf0165d7
SHLVL=0
HOME=/root
KAFKA_BOOTSTRAP=broker:29092
MONGO_PORT=27017
SPARK_BUFFER_SIZE=65536
GPG_KEY=F28C9C925C188C35E345614DEDA00CE834F0FC5C
PAGER=cat
SPARK_TGZ_ASC_URL=https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz.asc?action=download
MONGO_SPARK_VERSION=10.5.0
KAFKA_VERSION=4.0.1
SPARK_TGZ_URL=https://www.apache.org/dyn/closer.lua/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz?action=download
JAVA_VERSION=jdk-17.0.16+8
AWS_SDK_V1_VERSION=1.12.793
AWS_SDK_V2_VERSION=2.40.13
FORCE_COLOR=1
HADOOP_VERSION=3.4.0
MONGO_INITDB_ROOT_PASSWORD=password
SPARK_VERSION=4.0.1
TERM=xterm-color
MONGO_APP_DB=yelp
PATH=/opt/java/openjdk/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
MONGO_AUTH_DB=admin
LANG=en_US.UTF-8
CLICOLOR_FORCE=1
DELTA_VERSION=4.0.0
SCALA_VERSION=2.13
GIT_PAGER=cat
JAVA_HOME=/opt/java/op

In [3]:
spark.sql("SELECT 1 AS ok").show()

[Stage 0:>                                                          (0 + 1) / 1]

+---+
| ok|
+---+
|  1|
+---+



In [4]:
df = (spark.read.format("mongodb")
      .option("database", "test_db")
      .option("collection", "test_collection")
      .load())

df.show(5, truncate=False)

+------------------------+----------+----------------------------+------+---------+---------+---------------------------------------------------------------------------------+
|_id                     |created_at|name                        |owner |projectId|status   |tasks                                                                            |
+------------------------+----------+----------------------------+------+---------+---------+---------------------------------------------------------------------------------+
|6947bd5ef3d7a9a91d8de666|2024-01-10|Customer Analytics Dashboard|Alice |P001     |active   |[{T1, Data Cleaning, Bob, false}, {T2, ETL Pipeline Setup, Charlie, true}]       |
|6947bd5ef3d7a9a91d8de667|2023-12-02|Inventory Forecasting       |David |P002     |completed|[{T1, Collect SKU Data, Erika, true}, {T2, Train ML Model, Frank, true}]         |
|6947bd5ef3d7a9a91d8de668|2024-02-20|Website Redesign            |George|P003     |on_hold  |[{T1, UI Wireframes, Helen,

In [6]:
(df.write.format("mongodb")
 .option("database", "yelp")
 .option("collection", "spark_test")
 .mode("append")
 .save())
